In [1]:
import pickle
import pandas as pd
from agents.investigator import FraudInvestigationOrchestrator
from agents.tool_execution import ToolExecutor


In [2]:
def load_models():
    """Load trained LightGBM model and calibrator"""
    
    print("[SETUP] Loading trained models...")
    
    with open('model/lightgbm_model.pkl', 'rb') as f:
        model = pickle.load(f)
    
    with open('model/calibrator.pkl', 'rb') as f:
        calibrator = pickle.load(f)
    
    with open('model/shap_explainer.pkl', 'rb') as f:
        shap_explainer = pickle.load(f)
    
    with open('model/feature_names.pkl', 'rb') as f:
        feature_names = pickle.load(f)
    
    print(f"✓ Model loaded: {model.num_trees()} trees")
    print(f"✓ Features: {len(feature_names)}")
    
    return {
        'model': model,
        'calibrator': calibrator,
        'shap_explainer': shap_explainer,
        'feature_names': feature_names
    }


def load_data():
    """Load transaction data and features"""
    
    print("[SETUP] Loading transaction data...")
    
    # Load prepared data (from Notebook 01)
    with open('data/ieee_prepared.pkl', 'rb') as f:
        data_dict = pickle.load(f)
    
    # Load features (from Notebook 02)
    with open('data/features_test.pkl', 'rb') as f:
        X_test, y_test, feature_names = pickle.load(f)
    
    # Combine all transactions for tool context
    all_transactions = pd.concat([
        data_dict['train'],
        data_dict['val'],
        data_dict['test']    ])
    
    print(f"✓ Test set: {X_test.shape[0]} transactions")
    print(f"✓ Historical data: {len(all_transactions)} transactions")
    
    print(f"✓ Feature names: {len(feature_names)} features")
    # Ensure test features are in the same order as training features
    X_test = X_test[feature_names]

    print(f"Shape of X_test: {X_test.shape}")

    return {
        'X_test': X_test,
        'y_test': y_test,
        'all_transactions': all_transactions,
        'test_raw': data_dict['test']
    }

In [3]:
def score_with_ml(transaction_features, models):
    """Score transaction using trained XGBoost model"""
    
    model = models['model']
    calibrator = models['calibrator']
    # shap_explainer = models['shap_explainer']
    feature_names = models['feature_names']
    
    print(f"Transaction Features: {transaction_features}")
    # Raw prediction
    fraud_score_raw = model.predict(transaction_features)[0]
    try:
        fraud_score_raw = float(fraud_score_raw)
    except Exception:
        fraud_score_raw = float(str(fraud_score_raw).strip())
    
    # Calibrated prediction
    fraud_score_calibrated = calibrator.predict([fraud_score_raw])[0]
    try:
        fraud_score_calibrated = float(fraud_score_calibrated)
    except Exception:
        fraud_score_calibrated = float(str(fraud_score_calibrated).strip())

    # Load saved feature importance (< 1ms)
    importance_df = pd.read_csv('model/feature_importance.csv')
    top_features_df = importance_df.head(10)
    try:
        top_features_df['importance'] = top_features_df['importance'].astype(float)
    except Exception:
        top_features_df['importance'] = top_features_df['importance'].apply(lambda x: float(str(x).strip()))
    top_features = list(top_features_df[['feature', 'importance']].itertuples(index=False, name=None))
    
    # SHAP values
    # shap_values = shap_explainer.shap_values(feature_names)
    
    # Get top features
    # if isinstance(shap_values, list):
    #     shap_vals = shap_values[1]  # Class 1 (fraud)
    # else:
    #     shap_vals = shap_values
    
    # feature_impacts = list(zip(
    #     feature_names,
    #     shap_vals[0]  # First (only) sample
    # ))
    # feature_impacts.sort(key=lambda x: abs(x[1]), reverse=True)

    try:
        import shap
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(transaction_features)
    except ImportError:
        print("SHAP not available")
        
        print(f"\n[ML SCORING]")
        print(f"  Fraud Score (raw): {fraud_score_raw:.4f}")
        print(f"  Fraud Score (calibrated): {fraud_score_calibrated:.4f}")
        print(f"  Top features:")
        for feature_name, importance in top_features:
            try:
                importance = float(importance)
            except Exception:
                importance = float(str(importance).strip())
            print(f"    {feature_name}: {importance:.4f}")
    
    return {
        'fraud_score': fraud_score_calibrated,
        'fraud_score_raw': fraud_score_raw,
        'shap_values': shap_values if 'shap_values' in locals() else None,
        'feature_importance': top_features
    }

In [4]:
def investigate_transaction(transaction, all_transactions):
    """Run investigation tools for transaction"""
    
    print(f"\n[INVESTIGATION TOOLS]")
    
    tool_executor = ToolExecutor(all_transactions)
    tool_results_raw = tool_executor.execute_all(transaction)
    
    # Convert ToolResult objects to dicts for agent
    tool_results = {}
    for tool_name, result in tool_results_raw.items():
        tool_results[tool_name] = {
            'result': result.result,
            'confidence': result.confidence,
            'data_points': result.data_points,
            'error': result.error
        }
        
        status = "✓" if result.is_successful() else "✗"
        print(f"  {status} {tool_name}: confidence={result.confidence:.2f}, n={result.data_points}")
    
    return tool_results

In [5]:
 
def run_agents(transaction, ml_output, tool_results):
    """Run all 4 agents for complete assessment"""
    
    print(f"\n[AGENT REASONING]")
    
    orchestrator = FraudInvestigationOrchestrator()
    
    assessment = orchestrator.assess_transaction(
        transaction=transaction,
        ml_output=ml_output,
        tool_results=tool_results
    )
    
    print(f"  Risk Level: {assessment.risk_level}")
    print(f"  Risk Score: {assessment.risk_score:.2%}")
    print(f"  Confidence: {assessment.confidence:.0%}")
    print(f"  Decision: {assessment.decision}")
    print(f"  Execution Time: {assessment.execution_time_ms:.1f}ms")
    
    return assessment

In [6]:
# ============================================================================
# STEP 5: GENERATE FINAL REPORT
# ============================================================================
 
def generate_report(assessment, transaction, ml_output):
    """Generate final fraud assessment report"""
    
    print(f"\n{'='*80}")
    print("FRAUD ASSESSMENT REPORT")
    print(f"{'='*80}")
    
    print(f"\nTRANSACTION:")
    print(f"  ID: {transaction.get('TransactionID', 'N/A')}")
    print(f"  Amount: ${transaction.get('Amount', 0):.2f}")
    print(f"  Card: {transaction.get('card1', 'N/A')}")
    print(f"  Device: {transaction.get('DeviceInfo', 'N/A')}")
    
    print(f"\nRISK ASSESSMENT:")
    print(f"  Risk Level: {assessment.risk_level}")
    print(f"  Risk Score: {assessment.risk_score:.2%}")
    print(f"  Confidence: {assessment.confidence:.0%}")
    ml_contribution = assessment.ml_analysis.get('fraud_score', 0)
    try:
        ml_contribution = float(ml_contribution)
    except Exception:
        ml_contribution = float(str(ml_contribution).strip())
    print(f"  ML Contribution: {ml_contribution:.2%}")
    
    print(f"\nDECISION:")
    print(f"  Recommendation: {assessment.decision}")
    if assessment.decision == 'APPROVE':
        print(f"  → Transaction approved. Process normally.")
    elif assessment.decision == 'SOFT_DECLINE':
        print(f"  → Request 3D Secure verification before approval.")
    elif assessment.decision == 'REVIEW':
        print(f"  → Flag for manual fraud analyst review.")
    elif assessment.decision == 'BLOCK':
        print(f"  → DECLINE immediately. Block card if testing detected.")
    
    print(f"\nKEY FINDINGS:")
    key_signals = assessment.risk_assessment.get('key_signals', [])
    for signal in key_signals:
        print(f"  • {signal}")
    
    print(f"\nCUSTOMER EXPLANATION:")
    for line in assessment.customer_explanation.split('\n'):
        print(f"  {line}")
    
    print(f"\n{'='*80}\n")
    
    return {
        'decision': assessment.decision,
        'risk_level': assessment.risk_level,
        'risk_score': assessment.risk_score,
        'confidence': assessment.confidence,
        'key_findings': key_signals,
        'customer_explanation': assessment.customer_explanation
    }

In [7]:
def transform_features_for_scoring(X_test, feature_names):
    """Ensure features match training format"""
    X_test = X_test[feature_names].copy()  # Select right columns
    X_test = X_test.fillna(0)              # Fill NaN
    return X_test

In [ ]:


def main():
    """Run complete fraud detection assessment"""
    
    print("\n" + "="*80)
    print("FRAUD DETECTION AGENT SYSTEM - END-TO-END EXAMPLE")
    print("="*80)
    
    # Step 1: Load models and data
    models = load_models()
    data = load_data()
    

    X_test = transform_features_for_scoring(data['X_test'], models['feature_names'])

    # Step 2: Select a test transaction
    # Let's pick one that's likely fraud (to make demo interesting)
    test_idx = 0
    transaction_features = X_test.iloc[test_idx:test_idx+1]
    transaction_raw = data['test_raw'].iloc[test_idx]

    
    print(f"\n[TRANSACTION] Analyzing transaction {test_idx}...")
    print(f"  Amount: ${transaction_raw.get('TransactionAmt', 0):.2f}")
    print(f"  Actual label: {'FRAUD' if transaction_raw.get('isFraud') == 1 else 'LEGITIMATE'}")
    
    # Step 3: Score with ML
    # ml_output = score_with_ml(transaction_features, models)

    ml_output = score_with_ml(transaction_features, models)
    try:
        ml_output['fraud_score'] = float(ml_output['fraud_score'])
    except Exception:
        ml_output['fraud_score'] = float(str(ml_output['fraud_score']).strip())

    print(f"Score: {ml_output['fraud_score']:.4f}")
    print(f"Fraud Score datatype: {type(ml_output['fraud_score'])}")

    
    # Step 4: Run investigation tools
    tool_results = investigate_transaction(
        transaction_raw,
        data['all_transactions']
    )
    
    # Step 5: Run all 4 agents
    assessment = run_agents(
        transaction=transaction_raw.to_dict(),
        ml_output=ml_output,
        tool_results=tool_results
    )
    
    # Step 6: Generate report
    report = generate_report(assessment, transaction_raw, ml_output)
    
    # Return results
    return {
        'assessment': assessment,
        'report': report,

        'transaction': transaction_raw.to_dict(), 

        'ml_output': ml_output    }

In [9]:
def batch_assess(n_transactions=10):
    """Run assessment on multiple transactions"""
    
    print(f"\n[BATCH] Assessing {n_transactions} transactions...\n")
    
    models = load_models()
    data = load_data()
    
    results = []
    
    for i in range(min(n_transactions, len(data['X_test']))):
        print(f"[{i+1}/{n_transactions}] Transaction {i}...", end=" ")
        
        try:
            transaction_features = data['X_test'].iloc[i:i+1]
            transaction_raw = data['test_raw'].iloc[i]
            
            ml_output = score_with_ml(transaction_features, models)
            tool_results = investigate_transaction(transaction_raw, data['all_transactions'])
            assessment = run_agents(transaction_raw.to_dict(), ml_output, tool_results)
            
            results.append({
                'idx': i,
                'decision': assessment.decision,
                'risk_score': assessment.risk_score,
                'actual': transaction_raw.get('isFraud', -1)
            })
            
            print(f"→ {assessment.decision} (risk={assessment.risk_score:.2%})")
        
        except Exception as e:
            print(f"ERROR: {e}")
    
    # Summary
    print(f"\n{'='*80}")
    print("BATCH RESULTS SUMMARY")
    print(f"{'='*80}")
    
    df_results = pd.DataFrame(results)
    
    print(f"\nDecision Distribution:")
    print(df_results['decision'].value_counts())
    
    print(f"\nRisk Score Distribution:")
    print(df_results['risk_score'].describe())
    
    print(f"\nAccuracy (vs actual fraud label):")
    # Rough accuracy: BLOCK/REVIEW for actual fraud, APPROVE for legitimate
    df_results['predicted_fraud'] = df_results['decision'].isin(['BLOCK', 'REVIEW'])
    df_results['actual_fraud'] = df_results['actual'] == 1
    accuracy = (df_results['predicted_fraud'] == df_results['actual_fraud']).mean()
    print(f"  Accuracy: {accuracy:.1%}")
 

In [10]:
if __name__ == "__main__":
    # Run single transaction example
    results = main()
    
    # Or run batch on multiple transactions:
    # batch_assess(n_transactions=20)
    
    print("\n✓ Assessment complete!")
    print(f"Final decision: {results['report']['decision']}")
 


FRAUD DETECTION AGENT SYSTEM - END-TO-END EXAMPLE
[SETUP] Loading trained models...
✓ Model loaded: 843 trees
✓ Features: 460
[SETUP] Loading transaction data...
✓ Test set: 87292 transactions
✓ Historical data: 590540 transactions
✓ Feature names: 460 features
Shape of X_test: (87292, 460)

[TRANSACTION] Analyzing transaction 0...
  Amount: $0.00
  Actual label: LEGITIMATE
Transaction Features:         TransactionDT  TransactionAmt  ProductCD  card1  card2  card3  card4  \
502264       13175958            92.0          0   2408  571.0  150.0      2   

        card5  card6  addr1  ...  card_std_amount  card_max_amount  \
502264  226.0      1  123.0  ...        68.911192           478.48   

        device_num_cards  device_fraud_rate  device_mean_amount  \
502264                 0                0.0                 0.0   

        email_fraud_rate  email_mean_amount  address_fraud_rate  \
502264               0.0                0.0            0.026587   

        address_num_cards  h

/var/folders/s4/4mzgqp6168x7mrlhyvrb6nd00000gn/T/ipykernel_52654/2470282802.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_features_df['importance'] = top_features_df['importance'].astype(float)
/Users/vijaymendiratta/Desktop/fraud-investigation-engine/.venv/lib/python3.9/site-packages/shap/explainers/_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Score: 0.0007
Fraud Score datatype: <class 'float'>

[INVESTIGATION TOOLS]
  ✗ velocity: confidence=0.00, n=0
  ✗ device: confidence=0.00, n=0
  ✗ email: confidence=0.70, n=0
  ✗ address: confidence=0.00, n=0

[AGENT REASONING]
Fraud Score datatype in orchestrator: <class 'float'>
Fraud Score value in orchestrator: 0.0007493443237167478
  Risk Level: HIGH
  Risk Score: 78.00%
  Confidence: 88%
  Decision: SOFT_DECLINE
  Execution Time: 0.1ms

FRAUD ASSESSMENT REPORT

TRANSACTION:
  ID: 3489264
  Amount: $0.00
  Card: 2408
  Device: nan

RISK ASSESSMENT:
  Risk Level: HIGH
  Risk Score: 78.00%
  Confidence: 88%
  ML Contribution: 0.07%

DECISION:
  Recommendation: SOFT_DECLINE
  → Request 3D Secure verification before approval.

KEY FINDINGS:
  • velocity
  • device

CUSTOMER EXPLANATION:
  We need to verify this is you. We noticed some unusual activity on your account, including velocity, device. Please complete 3D Secure verification to approve this transaction.



✓ Assessment comple

In [ ]:
data_dict = load_data()

[SETUP] Loading transaction data...


✓ Test set: 87292 transactions
✓ Historical data: 590540 transactions


In [ ]:
# compare train and valid test categorical feature names

def compare_categorical_features(train_df, test_df, categorical_cols):
    """Compare unique values of categorical features between train and test sets"""
    for col in categorical_cols:
        train_unique = set(train_df[col].dropna().unique())
        test_unique = set(test_df[col].dropna().unique())
        
        only_in_train = train_unique - test_unique
        only_in_test = test_unique - train_unique
        
        print(f"\nFeature: {col}")
        print(f"  Unique in Train: {len(train_unique)}")
        print(f"  Unique in Test: {len(test_unique)}")
        print(f"  Only in Train: {len(only_in_train)}")
        print(f"  Only in Test: {len(only_in_test)}")
        
        if only_in_train:
            print(f"    Examples only in Train: {list(only_in_train)[:5]}")
        if only_in_test:
            print(f"    Examples only in Test: {list(only_in_test)[:5]}")